# 🏡 FollonicaVacanze — Test backend in locale

Questo notebook testa le funzioni Python del backend **senza bisogno di un server web**.
Esegui le celle in ordine. Prima di iniziare, assicurati di avere il file `.env.local` nella root del progetto.

## Setup — carica le variabili d'ambiente

In [ ]:
# Installa python-dotenv se non ce l'hai
# Esegui questa cella solo la prima volta
import subprocess
subprocess.run(['pip', 'install', 'python-dotenv'], capture_output=True)
print('✅ python-dotenv pronto')

In [ ]:
import sys
import os
import json
from pathlib import Path
from dotenv import load_dotenv

# Trova la root del progetto (cartella che contiene .env.local)
# Adatta questo percorso alla tua struttura di cartelle
PROJECT_ROOT = Path('C:/Users/2900_manganelli/Documents/follonica-vacanze')

# Carica .env.local
env_file = PROJECT_ROOT / '.env.local'
if env_file.exists():
    load_dotenv(env_file)
    print(f'✅ .env.local caricato da {env_file}')
else:
    print(f'⚠️  File .env.local non trovato in {PROJECT_ROOT}')
    print('   Crea il file con RESEND_API_KEY e OWNER_EMAIL')

# Aggiungi il progetto al path Python
sys.path.insert(0, str(PROJECT_ROOT))

# Verifica le variabili
resend_key = os.environ.get('RESEND_API_KEY', '')
owner_email = os.environ.get('OWNER_EMAIL', 'info@follonicavacanze.com')
print(f'RESEND_API_KEY: {"✅ configurata" if resend_key else "❌ mancante"}')
print(f'OWNER_EMAIL:    {owner_email}')

---
## TEST 1 — Verifica che i file dati siano corretti

In [ ]:
DATA_DIR = PROJECT_ROOT / 'data'

with open(DATA_DIR / 'proprieta.json', encoding='utf-8') as f:
    proprieta = json.load(f)

with open(DATA_DIR / 'ical-feeds.json', encoding='utf-8') as f:
    ical_feeds = json.load(f)

print(f'✅ Trovati {len(proprieta)} appartamenti in proprieta.json')
print()
for apt_id, casa in proprieta.items():
    feeds = ical_feeds.get(apt_id, {})
    feeds_ok = [k for k, v in feeds.items() if v and 'XXXXX' not in v]
    feeds_ko = [k for k, v in feeds.items() if not v or 'XXXXX' in v]
    stato = '✅' if feeds_ok else '⚠️ '
    print(f"{stato} {apt_id}: {casa['nome']} | capacità {casa['capacita']} | "
          f"feed configurati: {feeds_ok or 'NESSUNO — aggiorna ical-feeds.json'}")

---
## TEST 2 — Parser iCal

In [ ]:
# Importa le funzioni dal backend
from api.check_availability import parse_ical, fetch_ical, dates_overlap
from datetime import date

# Test con un file iCal di esempio (formato standard)
ical_esempio = """\
BEGIN:VCALENDAR
VERSION:2.0
BEGIN:VEVENT
DTSTART;VALUE=DATE:20260719
DTEND;VALUE=DATE:20260726
SUMMARY:Prenotazione confermata
END:VEVENT
BEGIN:VEVENT
DTSTART;VALUE=DATE:20260802
DTEND;VALUE=DATE:20260809
SUMMARY:Reserved
END:VEVENT
END:VCALENDAR
"""

eventi = parse_ical(ical_esempio)
print(f'✅ Trovati {len(eventi)} eventi nel calendario di esempio:')
for start, end in eventi:
    print(f'   📅 {start} → {end}')

print()
# Test sovrapposizione date
req_start = date(2026, 7, 19)
req_end   = date(2026, 7, 26)
for start, end in eventi:
    overlap = dates_overlap(start, end, req_start, req_end)
    print(f'   Periodo {start}→{end} vs richiesta {req_start}→{req_end}: '
          f'{"⚠️  SOVRAPPOSTO (occupato)" if overlap else "✅ Libero"}')

---
## TEST 3 — Verifica disponibilità (con feed iCal reali)

In [ ]:
from api.check_availability import is_disponibile, fetch_ical

# ✏️  Modifica queste date per il tuo test
CHECKIN  = '2026-07-19'  # deve essere un sabato
CHECKOUT = '2026-07-26'  # deve essere un sabato
OSPITI   = 4

req_start = date.fromisoformat(CHECKIN)
req_end   = date.fromisoformat(CHECKOUT)

print(f'🔍 Cerco appartamenti liberi dal {CHECKIN} al {CHECKOUT} per {OSPITI} ospiti')
print('-' * 60)

disponibili = []
for apt_id, casa in proprieta.items():
    if casa['capacita'] < OSPITI:
        print(f'⏭️  {apt_id}: capacità {casa["capacita"]} < {OSPITI} ospiti richiesti')
        continue
    
    libero = is_disponibile(apt_id, req_start, req_end, ical_feeds)
    stato  = '✅ DISPONIBILE' if libero else '❌ Occupato'
    print(f'{stato}  {apt_id}: {casa["nome"]}') 
    if libero:
        disponibili.append(apt_id)

print()
print(f'📊 Risultato: {len(disponibili)} appartamenti disponibili: {disponibili}')

---
## TEST 4 — Anteprima email (senza inviarla)

In [ ]:
from api.send_quote import email_proprietario, email_cliente, config
from IPython.display import HTML, display

# Dati di test
payload_test = {
    'nome':      'Mario',
    'cognome':   'Rossi',
    'email':     'mario.rossi@email.it',
    'telefono':  '+39 333 1234567',
    'note':      'Preferisco piano alto se possibile',
    'adulti':    2,
    'bambini':   1,
    'checkin':   '2026-07-19',
    'checkout':  '2026-07-26',
    'appartamento': {
        'id':   'rif003',
        'nome': 'Bilocale con Terrazzo Vista Mare',
        'url':  'https://www.follonicavacanze.com/appartamenti-in-affitto/appartamento-rif-003.html'
    }
}

cfg = config()

print('📧 ANTEPRIMA: email al proprietario')
print('=' * 50)
display(HTML(email_proprietario(payload_test, cfg)))

In [ ]:
print('📧 ANTEPRIMA: email di conferma al cliente')
print('=' * 50)
display(HTML(email_cliente(payload_test, cfg)))

---
## TEST 5 — Invio email reale via Resend
> ⚠️  Questa cella invia **email vere**. Eseguila solo quando sei pronto.

In [ ]:
from api.send_quote import send_email

resend_key  = os.environ.get('RESEND_API_KEY', '')
from_addr   = os.environ.get('FROM_ADDRESS', 'noreply@follonicavacanze.com')
# ✏️  Cambia con la tua email per il test
TEST_EMAIL  = 'tuaemail@test.it'

if not resend_key:
    print('❌ RESEND_API_KEY non configurata in .env.local')
else:
    ok, risposta = send_email(
        resend_key,
        from_addr,
        TEST_EMAIL,
        '✅ Test email — FollonicaVacanze',
        '<h2>Test riuscito!</h2><p>Il backend email funziona correttamente.</p>'
    )
    if ok:
        print(f'✅ Email inviata a {TEST_EMAIL}')
        print(f'   Risposta Resend: {risposta}')
    else:
        print(f'❌ Errore invio: {risposta}')
        print('   Verifica che il dominio mittente sia verificato su resend.com')

---
## TEST 6 — Simulazione completa della chiamata API

In [ ]:
# Simula esattamente quello che fa il frontend quando l'utente
# inserisce le date e preme 'Verifica disponibilità'
import urllib.request
import json

# Se hai avviato un server locale (vedi sotto), testa così:
# BASE_URL = 'http://localhost:8000'

# Altrimenti, testa le funzioni direttamente:
from api.check_availability import is_disponibile
from datetime import date

richiesta = {
    'checkin':  '2026-07-19',
    'checkout': '2026-07-26',
    'ospiti':    4
}

req_start = date.fromisoformat(richiesta['checkin'])
req_end   = date.fromisoformat(richiesta['checkout'])

risposta = {
    'disponibili': [
        {'id': apt_id, **casa}
        for apt_id, casa in proprieta.items()
        if casa['capacita'] >= richiesta['ospiti']
        and is_disponibile(apt_id, req_start, req_end, ical_feeds)
    ]
}

print('📤 Richiesta:')
print(json.dumps(richiesta, indent=2))
print()
print('📥 Risposta API:')
print(json.dumps(risposta, indent=2, ensure_ascii=False))